# 03 Base table and data quality
Joins the tract-level Overture counts from notebook 02 with the ACS housing units and the Census spine (land and water area, population, urban share), turns "found nothing" into zeros on purpose, adds size-free features and checks the table is consistent. Saves `eda_outputs/base_table.parquet`.

**Design choice that runs through the project.** The base table holds *structural* information (size, people, homes) and Overture's own counts. Vulnerability, hazard, tribal and rurality variables are kept out.

In [1]:
from common import *
t = pd.read_parquet(OUT / "tract_overture.parquet")
acs = pd.concat([pd.read_parquet(REF / r / f"{r}-census-acs-housing.parquet", columns=["GEOID", "housing_units"]) for r in REG])
sp = pd.read_parquet(tracts_parquet("census-tract-table"), columns=["GEOID", "STATEFP", "COUNTYFP", "state_usps", "ALAND", "AWATER", "pop_total", "pct_urban", "ur_class"])
for d in (acs, sp): d["GEOID"] = d.GEOID.astype(str)
b = t.merge(acs, on="GEOID", how="left").merge(sp, on="GEOID", how="left")
print(b.shape, "| rows without ACS:", int(b.housing_units.isna().sum()), "| rows without spine:", int(b.pop_total.isna().sum()))
print(b[["housing_units", "ALAND", "AWATER", "pop_total", "pct_urban"]].describe().T.to_string())

(9794, 69) | rows without ACS: 0 | rows without spine: 0
                  count            mean             std   min           25%           50%            75%               max
housing_units 9,794.000       1,746.679         807.701 0.000     1,219.000     1,645.000      2,144.000        13,827.000
ALAND         9,794.000 108,860,288.624 428,551,799.458 0.000 2,049,794.250 3,965,891.000 21,265,709.000 8,892,508,042.000
AWATER        9,794.000   2,099,392.579  17,974,615.948 0.000         0.000     2,927.000    168,048.750   866,385,073.000
pop_total     9,794.000       4,127.766       1,902.062 0.000     2,826.250     3,901.000      5,118.750        30,199.000
pct_urban     9,767.000           0.800           0.369 0.000         0.873         1.000          1.000             1.000


## 1. Missing counts become zeros
After the join, a tract with no road, place or building has NaN. For counts and lengths that means zero, so they are filled with 0. Medians, means and shares of an empty set stay NaN because "no buildings" has no median area. Each fill is recorded so later steps can still tell "zero" from "not measured".

In [2]:
count_cols = [c for c in t.columns if c.startswith(("bldg_n", "road_km", "road_n", "ru_", "rail_", "infra_", "poi_n"))] + ["bldg_area_m2"]
keep_nan = ["bldg_area_median_m2", "bldg_year_median", "bldg_src_n", "poi_conf_mean", "poi_src_n"]
count_cols = [c for c in count_cols if c not in keep_nan]
filled = b[count_cols].isna().sum()
print("NaN filled with 0 (columns with any):"); print(filled[filled > 0].sort_values(ascending=False).to_string())
b[count_cols] = b[count_cols].fillna(0)
b["bldg_src_n"] = b.bldg_src_n.fillna(0); b["poi_src_n"] = b.poi_src_n.fillna(0)
print("\nremaining NaN:", b.isna().sum()[lambda s: s > 0].to_dict())

NaN filled with 0 (columns with any):
road_km_trunk           7932
road_km_motorway        5921
rail_km                 5763
rail_n_pieces           5763
ru_km_track             5372
infra_n                 4886
infra_n_airport         4886
infra_n_transit         4886
infra_n_busstop         4886
road_km_primary         3894
road_km_unclassified    3835
ru_km_nonmotor           830
road_km_secondary        687
ru_km_unknown            433
road_km_tertiary         386
road_km_hwy              136
road_n_hwy_pieces        136
road_km_residential       40
ru_km_service              6
bldg_n_lt30m2              2
bldg_n_upd_ge2024          2
bldg_n_other               2
bldg_n_subtype             2
bldg_n_name                2
bldg_n_osm                 2
bldg_n_msft                2
bldg_n_google              2
bldg_n_esri                2
bldg_n_class               2
bldg_n_floors              2
bldg_n_ge300m2             2
bldg_n_height              2
bldg_n                     2
road_

## 2. Structural features and size-free rates
Land area comes from the Census ALAND field; the geodesic polygon area is kept as a cross-check. Densities and rates are size-free, so tracts of very different size can be compared.

In [3]:
b["land_km2"] = b.ALAND / 1e6
b["water_km2"] = b.AWATER / 1e6
b["water_share"] = b.AWATER / (b.ALAND + b.AWATER)
b["hu_per_km2"] = b.housing_units / b.land_km2.where(b.land_km2 > 0)
b["pop_per_km2"] = b.pop_total / b.land_km2.where(b.land_km2 > 0)
b["pop_per_hu"] = b.pop_total / b.housing_units.where(b.housing_units > 0)
b["bldg_per_km2"] = b.bldg_n / b.tract_area_km2
b["bldg_per_hu"] = b.bldg_n / b.housing_units.where(b.housing_units > 0)
b["bldg_cover_share"] = b.bldg_area_m2 / (b.tract_area_km2 * 1e6)
b["road_hwy_km_per_km2"] = b.road_km_hwy / b.tract_area_km2
b["road_km_per_km2"] = b.road_km_all / b.tract_area_km2
b["poi_per_km2"] = b.poi_n / b.tract_area_km2
b["poi_per_bldg"] = b.poi_n / b.bldg_n.where(b.bldg_n > 0)
b["poi_per_1000pop"] = 1000 * b.poi_n / b.pop_total.where(b.pop_total > 0)
b["geo_vs_aland"] = b.tract_area_km2 / b.land_km2.where(b.land_km2 > 0)
print(b.groupby("region")[["land_km2", "water_share", "geo_vs_aland"]].median().loc[REG].to_string())
print("\ngeodesic polygon area / ALAND: share of tracts with ratio in [0.95, 1.5]:", round(b.geo_vs_aland.between(.95, 1.5).mean(), 3))
print("polygon area below land area (ratio < 0.95):", int((b.geo_vs_aland < .95).sum()), "| land area zero:", int((b.land_km2 == 0).sum()))

                  land_km2  water_share  geo_vs_aland
region                                               
maricopa-az          2.714        0.000         1.000
northern-ca          4.773        0.000         1.000
eastern-ok          10.221        0.002         1.002
south-central-tx     3.704        0.001         1.001
eastern-wa          10.667        0.001         1.001

geodesic polygon area / ALAND: share of tracts with ratio in [0.95, 1.5]: 0.988
polygon area below land area (ratio < 0.95): 0 | land area zero: 2


**Takeaway:** The geodesic polygon area is never smaller than the Census land area. For 98.8% of tracts the ratio is between 0.95 and 1.5; the 116 tracts above 1.5 all have a water share of 34% to 93% (median 45%), as expected when the polygon includes water, and two tracts have zero land. Median land area per tract varies by region, from 2.7 km2 in Maricopa and 3.7 km2 in Texas to 10.2 km2 in Eastern OK, so counts are not comparable across tracts without a size term.

## 3. Edge cases that need an explicit rule
The write-up asks how zero-population, water-dominated and no-Overture-data tracts were handled. Count them, by region, so the rules are based on size and not on a guess.

In [4]:
edge = pd.DataFrame({
    "zero_population": b.pop_total == 0, "zero_housing_units": b.housing_units == 0,
    "water_share_gt_50": b.water_share > .5, "land_below_0.5km2": b.land_km2 < .5,
    "land_over_1000km2": b.land_km2 > 1000, "no_buildings": b.bldg_n == 0, "no_roads": b.road_n_pieces == 0,
    "no_places": b.poi_n == 0, "nothing_at_all": (b.bldg_n == 0) & (b.road_n_pieces == 0) & (b.poi_n == 0),
    "no_highway_class_road": b.road_km_hwy == 0, "region": b.region})
e = edge.groupby("region").sum().loc[REG]; e["tracts"] = b.region.value_counts()
print(e.T.to_string()); print("\ntotal over scored four:", e.loc[SCORED].sum().to_dict())
print("\nthe tracts with nothing at all:")
print(b.loc[edge.nothing_at_all, ["GEOID", "region", "land_km2", "water_share", "pop_total", "housing_units"]].to_string(index=False))

region                 maricopa-az  northern-ca  eastern-ok  south-central-tx  eastern-wa
zero_population                 11            0           1                14           1
zero_housing_units              27            0          10                43           2
water_share_gt_50                0            2           1                40           0
land_below_0.5km2                8            9          10               140           0
land_over_1000km2               61           36          28                94          49
no_buildings                     0            0           0                 2           0
no_roads                         0            0           0                 2           0
no_places                        0            0           0                 2           0
nothing_at_all                   0            0           0                 2           0
no_highway_class_road           33            7          28                64           5
tracts    

## 4. Internal consistency
Relations that must hold if the aggregation and joins are right.

In [5]:
chk = {
 "rows == 9794": len(b) == 9794, "GEOID unique": b.GEOID.is_unique,
 "no negative counts or lengths": bool((b[count_cols] >= 0).all().all()),
 "highway km <= all-road km": bool((b.road_km_hwy <= b.road_km_all + 1e-6).all()),
 "all-road km <= unfiltered km (+0.5%)": bool((b.road_km_all <= b.ru_km_all * 1.005 + 1e-6).all()),
 "source counts sum to buildings": bool((b[["bldg_n_osm", "bldg_n_msft", "bldg_n_google", "bldg_n_esri", "bldg_n_other"]].sum(axis=1) == b.bldg_n).all()),
 "conf08 <= conf05 <= poi_n": bool(((b.poi_n_conf08 <= b.poi_n_conf05) & (b.poi_n_conf05 <= b.poi_n)).all()),
 "open + closed <= poi_n": bool(((b.poi_n_open + b.poi_n_closed) <= b.poi_n).all()),
 "fire/ems/school/hospital <= poi_n": bool((b[["poi_n_fire", "poi_n_ems", "poi_n_school", "poi_n_hospital"]].sum(axis=1) <= b.poi_n).all()),
 "buildings with height <= buildings": bool((b.bldg_n_height <= b.bldg_n).all()),
 "highway pieces <= road pieces": bool((b.road_n_hwy_pieces <= b.road_n_pieces).all()),
}
for k, v in chk.items(): print(("OK  " if v else "FAIL"), k)
print("\ncorrelation of housing units with population (log1p, scored four):", round(np.corrcoef(np.log1p(b[b.region.isin(SCORED)].housing_units), np.log1p(b[b.region.isin(SCORED)].pop_total))[0, 1], 3))

OK   rows == 9794
OK   GEOID unique
OK   no negative counts or lengths
OK   highway km <= all-road km
OK   all-road km <= unfiltered km (+0.5%)
OK   source counts sum to buildings
OK   conf08 <= conf05 <= poi_n
OK   open + closed <= poi_n
OK   fire/ems/school/hospital <= poi_n
OK   buildings with height <= buildings
OK   highway pieces <= road pieces

correlation of housing units with population (log1p, scored four): 0.817


**Takeaway:** All ten consistency checks pass: highway km never exceeds total road km, supplier counts add to the building count, confidence and status subsets nest properly, and no count is negative. Housing units and population are strongly but not perfectly related (log correlation 0.82), so both carry information about size.

## 5. Save

In [6]:
b = b.sort_values(["region", "GEOID"]).reset_index(drop=True)
b.to_parquet(OUT / "base_table.parquet", index=False)
print(b.shape, "saved to", OUT / "base_table.parquet")
print(len(b.columns), "columns; structural/identity:", ["land_km2", "water_share", "housing_units", "pop_total", "tract_area_km2"])

(9794, 84) saved to C:\Users\prajw\OneDrive\Documents\5 sem\ada\MapEquityChallenge\eda_outputs\base_table.parquet
84 columns; structural/identity: ['land_km2', 'water_share', 'housing_units', 'pop_total', 'tract_area_km2']


**Takeaway:** `base_table.parquet` is saved with 9,794 rows and 84 columns. Structural variables are land, water, housing units, population and polygon area; Overture counts, lengths and quality fields sit alongside, and no strata (vulnerability, hazard, tribal) are in it.